# 02 - Baseline (Skenario A): semua fitur, random split

Tujuan: melatih Logistic Regression dan Random Forest pada **semua fitur numerik** dengan **stratified random split 80:20**, sebagai pembanding untuk Skenario B dan C.

Ketentuan yang dipakai:
- Kelas positif = **phishing** (`y_phishing = 1`). Precision, recall, dan F1 dihitung untuk phishing.
- Baris dengan URL duplikat dibuang sebelum pembagian data.
- Scaler (untuk Logistic Regression) hanya belajar dari data train, karena berada di dalam Pipeline.
- Metrik validasi dari 5-fold cross-validation di dalam data train. Data test dipakai satu kali untuk evaluasi akhir.
- Seed tetap: 42.

In [7]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent          # folder repo (notebook ada di folder notebooks/)
sys.path.insert(0, str(ROOT))

from src.preprocessing import (SEED, Y_COL, GROUP_COL, load_data, feature_sets,
                               split_random, split_group)
from src.evaluation import run_scenario, save_results

DATA_PATH = ROOT / "data" / "PhiUSIIL_Phishing_URL_Dataset.csv"
RESULTS_PATH = ROOT / "results" / "metrics.csv"
FIG_DIR = ROOT / "figures"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Muat data dan bersihkan

In [8]:
df, info = load_data(DATA_PATH)
print(info)
# Cek arti label: label 1 = legitimate, label 0 = phishing. y_phishing: 1 = phishing.
print(pd.crosstab(df["label"], df[Y_COL], rownames=["label (Kaggle)"], colnames=["y_phishing"]))

{'baris_awal': 235795, 'url_duplikat_dibuang': 425, 'baris_akhir': 235370}
y_phishing           0       1
label (Kaggle)                
0                    0  100520
1               134850       0


## 2. Fitur Skenario A
Semua kolom numerik, tanpa kolom teks/identitas (`FILENAME`, `URL`, `Domain`, `TLD`, `Title`) dan tanpa target.

In [9]:
features = feature_sets(df)["A"]
print(len(features), "fitur")
print(features)

50 fitur
['URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


## 3. Pembagian data (stratified random split 80:20)

In [10]:
train_idx, test_idx = split_random(df, test_size=0.2, seed=SEED)
print("train:", len(train_idx), "| test:", len(test_idx))
print("proporsi phishing di train:", round(df.loc[train_idx, Y_COL].mean(), 4))
print("proporsi phishing di test :", round(df.loc[test_idx, Y_COL].mean(), 4))

train: 188296 | test: 47074
proporsi phishing di train: 0.4271
proporsi phishing di test : 0.4271


## 4. Latih dan evaluasi (butuh beberapa menit)

In [11]:
res_A = run_scenario(df, "A", features, train_idx, test_idx,
                     split_name="random 80:20 (stratified)", seed=SEED,
                     grouped_cv=False, do_cv=True, fig_dir=FIG_DIR,
                     notes="semua fitur numerik; URL duplikat dibuang")
res_A.drop(columns=["scenario", "split", "features", "notes"]).round(5)

,model,n_features,seed,val_f1,val_roc_auc,test_f1,test_precision,test_recall,test_roc_auc,inference_ms_per_sample,n_train,n_test
0,LogisticRegression,50,42,0.99986,1.0,0.9998,1.0,0.9996,0.99999,0.00064,188296,47074
1,RandomForest,50,42,0.99999,1.0,1.0000,1.0,1.0000,1.00000,0.00599,188296,47074


## 5. Simpan hasil ke `results/metrics.csv`

In [12]:
save_results(res_A, RESULTS_PATH).round(5)

,scenario,model,split,features,n_features,seed,val_f1,val_roc_auc,test_f1,test_precision,test_recall,test_roc_auc,inference_ms_per_sample,n_train,n_test,notes
0,A,LogisticRegression,random 80:20 (stratified),semua,50,42,0.99986,1.0,0.9998,1.0,0.9996,0.99999,0.00064,188296,47074,semua fitur numerik; URL duplikat dibuang
1,A,RandomForest,random 80:20 (stratified),semua,50,42,0.99999,1.0,1.0000,1.0,1.0000,1.00000,0.00599,188296,47074,semua fitur numerik; URL duplikat dibuang


## Catatan untuk dijelaskan saat verifikasi
1. Kenapa kelas positifnya phishing, dan apa yang berubah kalau memakai label 1 (legitimate)?
2. Kenapa URL duplikat dibuang sebelum split?
3. Kenapa scaler ditaruh di dalam Pipeline?
4. Apa bedanya metrik validasi (cross-validation di train) dan metrik test?
5. Kalau skor di sini mendekati 1,0, apa kemungkinan penyebabnya? (lihat `URLSimilarityIndex` dan fitur HTML di notebook 03)